# AIOps Lab — Disk Utilisation Forecasting
### Predicting Disk Full Events Before They Happen
**AIOps Fundamentals Training | Microland**

---

## The Business Problem

One of the most common and costly IT incidents is a disk filling up unexpectedly,
causing application crashes, database corruption, and service outages.
Traditional monitoring only alerts you **when** the disk is already at 85% or 90% —
often giving you less than an hour to respond.

**AIOps Forecasting changes this completely.**

By learning the historical growth pattern of a disk, the ML model can predict
*days or weeks in advance* when the disk will reach a critical threshold.
This gives the operations team time to:
- Raise a proactive change request
- Schedule disk expansion during a maintenance window
- Archive old data before the disk fills
- Avoid the 3 AM emergency call entirely

---

## What This Notebook Does

This notebook generates **60 days of realistic disk utilisation data** for four server
with different growth patterns, uploads it to Elastic Cloud, and prepares it for
ML Forecasting in Kibana.

---

## Before You Start
1. Elastic Cloud deployment running (from the Day 1 lab)
2. Cloud ID and API Key ready
3. Running in Google Colab or Jupyter

> **No Python knowledge required.** Run each cell in order using the Play button.


## Step 1 — Install Required Libraries


In [ ]:
!pip install elasticsearch matplotlib --quiet
print('Libraries ready.')


## Step 2 — Configuration

Paste your Elastic Cloud credentials below.
The four servers and their growth patterns are pre-configured — review them
to understand what story each server is telling.


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────
CLOUD_ID   = 'Your Cloud Elastic ID'
API_KEY    = 'Your Elastic API'
INDEX_NAME = 'aiops-disk-forecast-index'

SERVER_NAME    = 'LOG-SRV-01'
START_PCT      = 0.60          # Starts at 60% sixty days ago
DAILY_GROWTH   = 0.004         # 0.4% stable growth per day
NOISE          = 0.001         # Very clean monotonic baseline line
DAYS_HISTORY   = 60            # 60 days of history for deep model confidence
SAMPLE_H       = 1
DISK_SIZE_GB   = 500
CRITICAL       = 0.90          # Target 90% threshold for alert

# Calculate metrics to show your trainees
total_history_growth = DAILY_GROWTH * DAYS_HISTORY
expected_today_pct = START_PCT + total_history_growth
days_to_critical_from_today = (CRITICAL - expected_today_pct) / DAILY_GROWTH

print(f'Server         : {SERVER_NAME}')
print(f'Start Value    : {START_PCT*100:.1f}% (60 days ago)')
print(f'Expected Today : {expected_today_pct*100:.1f}%')
print(f'Growth rate    : {DAILY_GROWTH*100:.1f}% per day')
print(f'Days until 90% : ~{days_to_critical_from_today:.1f} days from TODAY')

## Step 3 — Generate 60 Days of Disk Utilisation Data

Each server follows a realistic growth pattern with:
- **Natural hourly variation** (disk usage fluctuates slightly hour to hour)
- **Day/night pattern** (slightly higher during business hours due to active processes)
- **The configured growth trend** (linear, accelerating, or weekly spikes)

This gives the ML model realistic seasonal patterns to learn from —
the same patterns it would see from real Metricbeat data.


In [ ]:
import random
import math
from datetime import datetime, timedelta, timezone
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

random.seed(42)
es = Elasticsearch(cloud_id=CLOUD_ID, api_key=API_KEY)

# Delete and recreate index
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

mapping = {'mappings': {'properties': {
    '@timestamp':                 {'type': 'date'},
    'host.name':                  {'type': 'keyword'},
    'system.filesystem.used.pct': {'type': 'float'},
    'system.filesystem.used.bytes': {'type': 'long'},
    'system.filesystem.free':     {'type': 'long'},
    'system.filesystem.total':    {'type': 'long'},
    'metricset.name':             {'type': 'keyword'},
    'event.dataset':              {'type': 'keyword'},
}}}
es.indices.create(index=INDEX_NAME, body=mapping)
print(f'Index created: {INDEX_NAME}')

now        = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
start_time = now - timedelta(days=DAYS_HISTORY)
DISK_BYTES = DISK_SIZE_GB * 1024 * 1024 * 1024
total_samples = DAYS_HISTORY * 24  # hourly samples

events = []

for i in range(total_samples):
    ts          = start_time + timedelta(hours=i)
    day_elapsed = i / 24.0

    # Build clear linear growth trend up to the present hour
    used_pct = START_PCT + (DAILY_GROWTH * day_elapsed) + random.gauss(0, NOISE)
    used_pct = min(0.99, max(0.10, used_pct))

    used_bytes = int(used_pct * DISK_BYTES)
    free_bytes = DISK_BYTES - used_bytes

    events.append({
        '@timestamp':                 ts.isoformat(),
        'host.name':                  SERVER_NAME,
        'system.filesystem.used.pct': round(used_pct, 4),
        'system.filesystem.used.bytes': used_bytes,
        'system.filesystem.free':     free_bytes,
        'system.filesystem.total':    DISK_BYTES,
        'metricset.name':             'filesystem',
        'event.dataset':              'system.filesystem',
    })

# Upload
def gen_actions(docs, idx):
    for d in docs: yield {'_index': idx, '_source': d}

success, errors = bulk(es, gen_actions(events, INDEX_NAME),
                       chunk_size=500, raise_on_error=False)
es.indices.refresh(index=INDEX_NAME)

start_val = events[0]['system.filesystem.used.pct']
end_val   = events[-1]['system.filesystem.used.pct']

print(f'Uploaded {success:,} hourly samples')
print(f'   Start value : {start_val*100:.1f}%  ({start_time.strftime("%Y-%m-%d")})')
print(f'   End value   : {end_val*100:.1f}%  (Today\'s Baseline)')
print(f'   Growth seen : +{(end_val-start_val)*100:.1f}% over historical background')
print()
print('Data simulation finalized for your Machine Learning Lab.')
print(f'Run a 14-day forecast on mean(system.filesystem.used.pct) in Kibana to see the crossover point.')

## Step 4 — Visualise the growth



In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime

timestamps = [datetime.fromisoformat(e['@timestamp'].replace('Z', '+00:00'))
              for e in events]
pct_values = [e['system.filesystem.used.pct'] * 100 for e in events]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(timestamps, pct_values, color='#1B3A6B', linewidth=1.2, alpha=0.8)
ax.axhline(90, color='red', linestyle='--', linewidth=1.5, label='Critical (90%)')
ax.axhline(START_PCT * 100, color='#0D6E6E', linestyle=':', linewidth=1.5,
           label=f'Start value ({START_PCT*100:.0f}%)')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Disk Used %', fontsize=11)
ax.set_title(f'Disk Utilisation — {SERVER_NAME} | {DAYS_HISTORY} days of history\n'
             f'Growth rate: {DAILY_GROWTH*100:.1f}%/day | '
             f'Forecast target: when will it cross 90%?',
             fontsize=11, fontweight='bold')
ax.set_ylim(60, 100)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Chart shows a clean, clear upward trend.')
print(f'The ML model will learn this trend and project it forward.')
print(f'Expected forecast: disk crosses 90% in approximately ')

# Step 5--Create the Forecast in Elastic

In [ ]:
"""
Kibana > Stack Management > Data Views > Create data view
   Name          : Disk Forecast Lab
   Index pattern : aiops-disk-forecast-index
   Timestamp     : @timestamp
   Save

Kibana > Machine Learning > Anomaly Detection > Create Job
   Data view    : Disk Forecast Lab
   Time range   : Full (all 60 days)
   Job type     : Single Metric

   Function     : Mean           ← NOT High Mean
   Field        : system.filesystem.used.pct → Mean sub-field
   Bucket span  : 1h             ← type manually, do not click Estimate
   Job ID       : disk-forecast-job

   Advanced     → Enable model plot: ON   ← must be ON for forecast to work

   Click: Create job (wait ~90 seconds)

Single Metric Viewer:
   Click: Forecast
   Duration: 14d                 ← 14 days
"""